In [ ]:
import os
from getpass import getpass

import pandas as pd
import pymysql
from IPython.display import display

password = os.getenv("SALES_DB_PASSWORD")
if not password:
    password = getpass("MySQL password: ")

db_config = {
    "host": os.getenv("SALES_DB_HOST", "localhost"),
    "user": os.getenv("SALES_DB_USER", "root"),
    "password": password,
    "database": os.getenv("SALES_DB_NAME", "sales"),
}
del password


In [ ]:
def run_query(query):
    connection = pymysql.connect(**db_config)
    try:
        return pd.read_sql_query(query, connection)
    finally:
        connection.close()


In [ ]:
# Task 1: Monthly Revenue Trend — 2022
query = """
SELECT
    DATE_FORMAT(order_date, '%Y-%m') AS month,
    SUM(before_discount) AS total_revenue
FROM order_detail
WHERE YEAR(order_date) = 2022
GROUP BY DATE_FORMAT(order_date, '%Y-%m')
ORDER BY month;
"""

monthly_revenue = run_query(query)
display(monthly_revenue)


In [ ]:
# Task 2: Sales by Payment Method
query = """
SELECT
    paydet.payment_method,
    SUM(ordet.before_discount) AS total_sales,
    SUM(ordet.qty_ordered) AS total_quantity
FROM order_detail AS ordet
JOIN payment_detail AS paydet
    ON ordet.payment_id = paydet.id
GROUP BY paydet.payment_method
ORDER BY total_sales DESC;
"""

payment_sales = run_query(query)
display(payment_sales)


In [ ]:
# Task 3: Average Order Value — 2022
query = """
SELECT
    DATE_FORMAT(order_date, '%Y-%m') AS month,
    SUM(after_discount) / COUNT(DISTINCT id) AS AOV
FROM order_detail
WHERE YEAR(order_date) = 2022
GROUP BY DATE_FORMAT(order_date, '%Y-%m')
ORDER BY month;
"""

aov = run_query(query)
display(aov)


In [ ]:
# Task 4: Revenue Before and After Discount by Category
query = """
SELECT
    skudet.category,
    SUM(ordet.before_discount) AS revenue_before_discount,
    SUM(ordet.after_discount) AS revenue_after_discount,
    SUM(ordet.before_discount) - SUM(ordet.after_discount) AS discount_impact
FROM order_detail AS ordet
JOIN sku_detail AS skudet
    ON ordet.sku_id = skudet.id
GROUP BY skudet.category
ORDER BY revenue_before_discount DESC;
"""

discount_impact = run_query(query)
display(discount_impact)


In [ ]:
# Task 5: Top 10 Products with Largest Sales Decrease — 2021 vs 2022
query = """
WITH sales_2021 AS (
    SELECT skudet.sku_name, SUM(ordet.qty_ordered) AS sales_2021
    FROM order_detail AS ordet
    JOIN sku_detail AS skudet ON ordet.sku_id = skudet.id
    WHERE YEAR(ordet.order_date) = 2021
    GROUP BY skudet.sku_name
),
sales_2022 AS (
    SELECT skudet.sku_name, SUM(ordet.qty_ordered) AS sales_2022
    FROM order_detail AS ordet
    JOIN sku_detail AS skudet ON ordet.sku_id = skudet.id
    WHERE YEAR(ordet.order_date) = 2022
    GROUP BY skudet.sku_name
),
all_products AS (
    SELECT sku_name FROM sales_2021
    UNION
    SELECT sku_name FROM sales_2022
)
SELECT
    p.sku_name,
    COALESCE(s21.sales_2021, 0) AS sales_2021,
    COALESCE(s22.sales_2022, 0) AS sales_2022,
    COALESCE(s22.sales_2022, 0) - COALESCE(s21.sales_2021, 0) AS difference
FROM all_products AS p
LEFT JOIN sales_2021 AS s21 ON p.sku_name = s21.sku_name
LEFT JOIN sales_2022 AS s22 ON p.sku_name = s22.sku_name
WHERE COALESCE(s22.sales_2022, 0) - COALESCE(s21.sales_2021, 0) < 0
ORDER BY difference ASC
LIMIT 10;
"""

sales_decrease = run_query(query)
display(sales_decrease)


In [ ]:
# Task 6: Weekend vs Weekday Average Daily Sales — Q4 2022
monthly_q4_query = """
WITH daily_sales AS (
    SELECT DATE(order_date) AS sales_date,
           SUM(before_discount) AS daily_sales
    FROM order_detail
    WHERE order_date >= '2022-10-01'
      AND order_date < '2023-01-01'
    GROUP BY DATE(order_date)
),
classified_days AS (
    SELECT sales_date, daily_sales,
           DATE_FORMAT(sales_date, '%Y-%m') AS month,
           CASE WHEN DAYOFWEEK(sales_date) IN (1, 7)
                THEN 'Weekend' ELSE 'Weekday' END AS day_type
    FROM daily_sales
)
SELECT month, day_type, AVG(daily_sales) AS average_daily_sales
FROM classified_days
GROUP BY month, day_type
ORDER BY month, day_type;
"""

overall_q4_query = """
WITH daily_sales AS (
    SELECT DATE(order_date) AS sales_date,
           SUM(before_discount) AS daily_sales
    FROM order_detail
    WHERE order_date >= '2022-10-01'
      AND order_date < '2023-01-01'
    GROUP BY DATE(order_date)
),
classified_days AS (
    SELECT sales_date, daily_sales,
           CASE WHEN DAYOFWEEK(sales_date) IN (1, 7)
                THEN 'Weekend' ELSE 'Weekday' END AS day_type
    FROM daily_sales
)
SELECT day_type, AVG(daily_sales) AS overall_average_daily_sales
FROM classified_days
GROUP BY day_type
ORDER BY day_type;
"""

monthly_q4 = run_query(monthly_q4_query)
overall_q4 = run_query(overall_q4_query)
display(monthly_q4)
display(overall_q4)
